In [19]:
!pip install -q smolagents

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.8/149.8 kB 4.4 MB/s eta 0:00:00


In [22]:
from smolagents import DuckDuckGoSearchTool

## Dux Distributed Global Search

In [2]:
!pip install -q ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.6/41.6 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 52.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 69.3 MB/s eta 0:00:00:00:01


In [3]:
from ddgs import DDGS

In [18]:
results = DDGS().text(
    "most popular programming language",
    max_results=5,
    safesearch="on",
    backend="google",
)
print(results)

[{'title': 'TIOBE index - Wikipedia', 'href': 'https://en.wikipedia.org/wiki/TIOBE_index', 'body': 'The TIOBE programming community index is a measure of popularity of programming languages , created and maintained by TIOBE Software BV, based in Eindhoven, the Netherlands. TIOBE stands for The Importance of Being Earnest, the title of an 1895 comedy...'}, {'title': "The Most Popular Programming Languages in Today's Tech World", 'href': 'https://www.linkedin.com/pulse/most-popular-programming-languages-todays-tech-world-tekvaly-8v50f', 'body': 'Moreover, the popularity of a programming language often determines its relevance and demand in the job market. Here, we explore some of the most popular programming languages , offering a comprehensive overview of their significance, usage statistics...'}, {'title': 'The Top Programming Languages 2025 - IEEE Spectrum', 'href': 'https://spectrum.ieee.org/top-programming-languages-2025', 'body': 'As long as there’s enough training data, they’ll ge

In [33]:
# based on https://medium.com/@laurentkubaski/smolagents-duckduckgosearchtool-to-search-in-wikipedia-2578973bb131

class CustomDuckDuckGoSearchTool(DuckDuckGoSearchTool):
    name = "web_search"
    description = "Performs a web search for a query and returns a list of the top search results formatted as markdown with page titles and urls."
    inputs = {"query": {"type": "string", "description": "The search query to perform."}}
    output_type = "string"

    def __init__(self, max_results: int = 10, rate_limit: float | None = 1.0, backend: str = "auto", **kwargs):
        super().__init__(max_results=max_results, rate_limit=rate_limit, **kwargs)
        self.backend = backend # Add "backend" as new parameter

    def forward(self, query: str) -> str:
        self._enforce_rate_limit()
        results = self.ddgs.text(
            query=query,
            max_results=self.max_results,
            backend=self.backend) # there you go
        
        if len(results) == 0:
            raise Exception("No results found! Try a less restrictive/shorter query.")
        
        postprocessed_results = [
            f"{i+1}. [{result['title']}]({result['href']})\nBody: {result['body']}"
            for i, result in enumerate(results)
        ]

        return "## Search Results\n" + "\n".join(postprocessed_results)

In [34]:
tool = CustomDuckDuckGoSearchTool(
    max_results=5,
    rate_limit=1.0,
    backend="google")

result = tool(query='most popular programming language')
print(result)

## Search Results
1. [Most Popular Programming Languages 1965 - 2019 - YouTube](https://www.youtube.com/watch?v=Og847HVwRSI)
Body: Timeline of the most popular programming languages since 1965 to 2019. So far the most intense ranking I've ever done :) For recent years I ...
2. [The most popular programming languages in 2025 (and what that](https://www.zdnet.com/article/the-most-popular-programming-languages-in-2024-and-what-that-even-means/)
Body: The most popular programming languages in 2025 (and what that even means) ... PYPL: The PopularitY of Programming Language index derives data from ...
3. [PYPL PopularitY of Programming Language index](https://pypl.github.io/PYPL.html)
Body: The PYPL PopularitY of Programming Language Index is created by analyzing how often language tutorials are searched on Google : the more a language ...
4. [Most Popular Programming Languages in the World 2024: Top 16](https://bscholarly.com/most-popular-programming-languages/)
Body: Most Popular Programmi